# ПР №3 — Пайплайн предобработки, уровень B

Используется тот же **демонстрационный синтетический** датасет, что в ПР-2. Важно: все статистики иммутации и масштабирования вычисляются **только на train**.

## Стратегии для каждой группы

- `age`, `experience_years`, `monthly_income_som` — **медиана** (есть выбросы, среднее менее устойчиво).
- `household_size` — **медиана** (целочисленный счётчик, возможно искажение распределения; для этого учебного набора допустима медиана).
- `city`, `contract_type` — заполнение константой **`Неизвестно`** и one-hot (`handle_unknown='ignore'`).
- Для числовых переменных после заполнения — `StandardScaler`. Для исходных числовых выбросов масштабирование не защищает от аномалий: это ограничение выбранного подхода.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler

df=pd.read_csv('../data/dirty_employees.csv')
X=df.drop(columns='will_leave');y=df['will_leave']
num=X.select_dtypes(include='number').columns.tolist()
cat=X.select_dtypes(exclude='number').columns.tolist()
print('Числовые:',num,'\nКатегории:',cat)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
print('Train:',X_train.shape,'Test:',X_test.shape)

Числовые: ['age', 'experience_years', 'monthly_income_som', 'household_size'] 
Категории: ['city', 'contract_type']
Train: (384, 6) Test: (96, 6)


In [2]:
numeric=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
categorical=Pipeline([('imputer',SimpleImputer(strategy='constant',fill_value='Неизвестно')),('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
preprocessor=ColumnTransformer([('num',numeric,num),('cat',categorical,cat)],remainder='drop')
Xtr=preprocessor.fit_transform(X_train)
Xte=preprocessor.transform(X_test)  # НЕ fit_transform!
print('До:',X_train.shape,X_test.shape,'После:',Xtr.shape,Xte.shape)
print('Имена итоговых признаков:',preprocessor.get_feature_names_out().tolist())
print('Число NaN на train/test:',np.isnan(Xtr).sum(),np.isnan(Xte).sum())
assert Xtr.shape[1]==Xte.shape[1]
assert np.isfinite(Xtr).all() and np.isfinite(Xte).all()

До: (384, 6) (96, 6) После: (384, 16) (96, 16)
Имена итоговых признаков: ['num__age', 'num__experience_years', 'num__monthly_income_som', 'num__household_size', 'cat__city_Баткен', 'cat__city_Бишкек', 'cat__city_Каракол', 'cat__city_Нарын', 'cat__city_Неизвестно', 'cat__city_Ош', 'cat__city_Талас', 'cat__contract_type_Неизвестно', 'cat__contract_type_временный', 'cat__contract_type_постоянный', 'cat__contract_type_фриланс', 'cat__contract_type_частичная занятость']
Число NaN на train/test: 0 0


## Проверка отсутствия утечки

Обучение `fit_transform` вызвано **один раз на train**. Для test используется только `transform`. Ни медиана, ни среднее/дисперсия масштабирования, ни перечень категорий не вычисляются на test. Размерности совпадают благодаря единому one-hot кодировщику.

In [3]:
names=preprocessor.get_feature_names_out()
print('Учебная медиана возраста:',preprocessor.named_transformers_['num'].named_steps['imputer'].statistics_[num.index('age')])
print('Первые строки матрицы:')
display(pd.DataFrame(Xtr[:5],columns=names).iloc[:,:min(10,len(names))].round(2))
report=Path('../reports');report.mkdir(exist_ok=True)
pd.DataFrame({'group':['train','test'],'rows':[len(Xtr),len(Xte)],'columns':[Xtr.shape[1],Xte.shape[1]]}).to_csv(report/'shapes.csv',index=False)

Учебная медиана возраста: 38.0
Первые строки матрицы:


,num__age,num__experience_years,num__monthly_income_som,num__household_size,cat__city_Баткен,cat__city_Бишкек,cat__city_Каракол,cat__city_Нарын,cat__city_Неизвестно,cat__city_Ош
0,-0.89,-1.26,0.02,0.94,0.0,1.0,0.0,0.0,0.0,0.0
1,-1.04,-1.26,-0.24,1.45,0.0,0.0,0.0,1.0,0.0,0.0
2,1.47,1.82,-0.39,0.43,0.0,1.0,0.0,0.0,0.0,0.0
3,-0.89,-0.73,0.36,-0.08,0.0,1.0,0.0,0.0,0.0,0.0
4,1.03,0.59,-0.44,-0.58,0.0,1.0,0.0,0.0,0.0,0.0


## Вывод

Выбранный `ColumnTransformer` единообразно обрабатывает числа и категории. Даже если в тестовой части встретится новая категория, преобразование не сломается. Главный принцип — **не использовать статистики теста при обучении**. В дальнейшем имеет смысл отдельно проверить влияние аномальных значений на качество модели.